# 🔬 Thoth Model Generation Evaluation

This notebook evaluates the bilingual generation capability of trained Thoth models.

## Setup Requirements

Before running, ensure you have uploaded the following to Google Drive:

```
Google Drive/
└── thoth_eval/
    ├── model.py                 # Model definition
    ├── 7_generation_eval.py     # Generation script
    ├── 8_llm_scoring.py         # Scoring script
    ├── tokenizer/               # Tokenizer directory
    └── checkpoints/
        ├── r0v0_full_model.pt   # r0v0_full checkpoint
        ├── r2v7_model.pt        # r2v7 checkpoint
        └── r3v10_model.pt       # r3v10 checkpoint
```

### 📥 Cluster Commands to Prepare Files

```bash
# On cluster:
cd /scratch/$USER/thoth_project

# Rename checkpoints
cp models/experiments/r0v0_full/checkpoints/model.pt ./r0v0_full_model.pt
cp models/experiments/r2v7_xxx/checkpoints/model.pt ./r2v7_model.pt
cp models/experiments/r3v10_xxx/checkpoints/model.pt ./r3v10_model.pt

# Package tokenizer
tar -czvf tokenizer.tar.gz models/tokenizer/
```

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Paths

In [ ]:
import os
import sys

# Base path - MODIFY THIS to match your Google Drive structure
BASE_PATH = "/content/drive/MyDrive/thoth_eval"

# Add to Python path
sys.path.insert(0, BASE_PATH)

# Tokenizer path (shared by all models)
TOKENIZER_PATH = os.path.join(BASE_PATH, "tokenizer")

# Output directory
OUTPUT_DIR = os.path.join(BASE_PATH, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Thoth Model configurations
THOTH_MODELS = {
    "r0v0_full": {
        "checkpoint": os.path.join(BASE_PATH, "checkpoints", "r0v0_full_model.pt"),
        "description": "Full dataset, lr=1e-5, no scheduling"
    },
    "r2v7": {
        "checkpoint": os.path.join(BASE_PATH, "checkpoints", "r2v7_model.pt"),
        "description": "Medium dataset, lr=1e-4, warmup=1%"
    },
    "r3v10": {
        "checkpoint": os.path.join(BASE_PATH, "checkpoints", "r3v10_model.pt"),
        "description": "Medium dataset, lr=3e-4, warmup=3%, cosine"
    }
}

# LLM Scorer configurations
SCORER_MODELS = {
    "mistral": {"name": "mistralai/Mistral-7B-Instruct-v0.2", "memory_gb": 14},
    "croissant": {"name": "croissantllm/CroissantLLMChat-v0.1", "memory_gb": 14},
    "qwen": {"name": "Qwen/Qwen2.5-7B-Instruct", "memory_gb": 14},
    "llama": {"name": "meta-llama/Llama-3.1-8B-Instruct", "memory_gb": 16},
    "deepseek": {"name": "deepseek-ai/deepseek-llm-7b-chat", "memory_gb": 14},
    "phi": {"name": "microsoft/Phi-3-mini-4k-instruct", "memory_gb": 8}
}

# Verify files
print("Checking files...")
print(f"✓ Tokenizer: {os.path.exists(TOKENIZER_PATH)}")
print(f"✓ model.py: {os.path.exists(os.path.join(BASE_PATH, 'model.py'))}")
print()
print("Thoth Models:")
for name, cfg in THOTH_MODELS.items():
    exists = os.path.exists(cfg["checkpoint"])
    status = "✓" if exists else "✗"
    print(f"  {status} {name}: {exists}")

## 3. Install Dependencies

In [ ]:
!pip install -q transformers tokenizers tqdm accelerate bitsandbytes

## 4. Check GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {mem_gb:.1f} GB")
    
    # Recommend scorers based on memory
    print("\nRecommended scorers for your GPU:")
    for key, info in SCORER_MODELS.items():
        if mem_gb >= info["memory_gb"]:
            print(f"  ✓ {key} ({info['memory_gb']}GB)")
        else:
            print(f"  ⚠ {key} ({info['memory_gb']}GB) - may need 4-bit quantization")

---
# 🎯 Select Thoth Model to Evaluate

In [ ]:
#@title Select Thoth Model {display-mode: "form"}
#@markdown Choose which Thoth model to evaluate:

SELECTED_THOTH = "r0v0_full" #@param ["r0v0_full", "r2v7", "r3v10"]

MODEL_CONFIG = THOTH_MODELS[SELECTED_THOTH]
MODEL_PATH = MODEL_CONFIG["checkpoint"]

print(f"\n{'='*60}")
print(f"Selected Thoth Model: {SELECTED_THOTH}")
print(f"Description: {MODEL_CONFIG['description']}")
print(f"{'='*60}")

## 5. Load Selected Model

In [ ]:
from model import GPT2
from transformers import PreTrainedTokenizerFast

# Load tokenizer
print("Loading tokenizer...")
tokenizer = PreTrainedTokenizerFast.from_pretrained(TOKENIZER_PATH)
print(f"Vocab size: {tokenizer.vocab_size}")

# Load model
print(f"\nLoading model: {SELECTED_THOTH}...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPT2.load(MODEL_PATH).to(device)
model.eval()
print(f"Model parameters: {model.get_num_params():,}")

## 6. Test Single Generation

In [ ]:
def beam_search_generate(model, tokenizer, prompt, max_new_tokens=50, num_beams=5):
    """Simple beam search generation."""
    device = next(model.parameters()).device
    eos_id = tokenizer.convert_tokens_to_ids("<eos>")
    
    encoded = tokenizer(prompt, return_tensors="pt")
    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    
    beams = [(0.0, input_ids.clone(), attention_mask.clone())]
    completed = []
    
    for _ in range(max_new_tokens):
        all_candidates = []
        for score, seq, mask in beams:
            if seq[0, -1].item() == eos_id:
                completed.append((score, seq))
                continue
            
            max_seq_len = model.config["model"]["max_seq_len"]
            if seq.size(1) > max_seq_len:
                seq = seq[:, -max_seq_len:]
                mask = mask[:, -max_seq_len:]
            
            with torch.no_grad():
                logits, _ = model(seq, attention_mask=mask, labels=None)
            
            next_logits = logits[0, -1, :]
            log_probs = torch.log_softmax(next_logits, dim=-1)
            top_log_probs, top_indices = torch.topk(log_probs, num_beams)
            
            for i in range(num_beams):
                new_token = top_indices[i].unsqueeze(0).unsqueeze(0)
                new_score = score + top_log_probs[i].item()
                new_seq = torch.cat([seq, new_token], dim=1)
                new_mask = torch.cat([mask, torch.ones(1, 1, device=device)], dim=1)
                all_candidates.append((new_score, new_seq, new_mask))
        
        if not all_candidates:
            break
        
        all_candidates.sort(key=lambda x: x[0], reverse=True)
        beams = all_candidates[:num_beams]
        if len(completed) >= num_beams:
            break
    
    completed.extend([(s, seq) for s, seq, _ in beams])
    completed.sort(key=lambda x: x[0], reverse=True)
    return tokenizer.decode(completed[0][1][0], skip_special_tokens=False)

# Test
test_prompts = [
    "<en> Hello, how are you? <fr>",
    "<fr> Bonjour, comment allez-vous? <en>",
]

print(f"Testing {SELECTED_THOTH}:")
for prompt in test_prompts:
    output = beam_search_generate(model, tokenizer, prompt)
    print(f"\nInput:  {prompt}")
    print(f"Output: {output}")

## 7. Run Full Generation

In [ ]:
generation_output = os.path.join(OUTPUT_DIR, f"{SELECTED_THOTH}_generations.json")
print(f"Output: {generation_output}")

!python "{BASE_PATH}/7_generation_eval.py" \
    --model_path "{MODEL_PATH}" \
    --tokenizer_path "{TOKENIZER_PATH}" \
    --output_path "{generation_output}" \
    --device 0

## 8. View Generated Samples

In [ ]:
import json

with open(generation_output, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total: {data['num_prompts']} samples")
for r in data["results"][:5]:
    print(f"\n[{r['type']}] {r['input_prompt']}")
    print(f"  → {r['generated_only']}")

---
# 🎯 Select LLM Scorer

In [ ]:
#@title Select LLM Scorer {display-mode: "form"}
#@markdown Choose which LLM to use for scoring:

SELECTED_SCORER = "mistral" #@param ["mistral", "croissant", "qwen", "llama", "deepseek", "phi"]
USE_4BIT = True #@param {type:"boolean"}

scorer_info = SCORER_MODELS[SELECTED_SCORER]
print(f"\n{'='*60}")
print(f"Selected Scorer: {SELECTED_SCORER}")
print(f"Model: {scorer_info['name']}")
print(f"Memory: ~{scorer_info['memory_gb']}GB")
print(f"4-bit quantization: {USE_4BIT}")
print(f"{'='*60}")

## 9. Free Memory & Run Scoring

In [ ]:
# Free Thoth model memory before loading scorer
del model
torch.cuda.empty_cache()
print("Thoth model unloaded, memory freed")

In [ ]:
scoring_output = os.path.join(OUTPUT_DIR, f"{SELECTED_THOTH}_scored_{SELECTED_SCORER}.json")
quantize_flag = "--use_4bit" if USE_4BIT else ""

print(f"Scoring {SELECTED_THOTH} with {SELECTED_SCORER}...")
print(f"Output: {scoring_output}")

!python "{BASE_PATH}/8_llm_scoring.py" \
    --input_path "{generation_output}" \
    --output_path "{scoring_output}" \
    --scorer "{SELECTED_SCORER}" \
    --device 0 \
    {quantize_flag}

## 10. View Scoring Results

In [ ]:
if os.path.exists(scoring_output):
    with open(scoring_output, "r", encoding="utf-8") as f:
        scores = json.load(f)
    
    summary = scores["summary"]
    print(f"\n{'='*60}")
    print(f"SCORES: {SELECTED_THOTH} (scorer: {SELECTED_SCORER})")
    print(f"{'='*60}")
    
    if "overall_averages" in summary:
        avg = summary["overall_averages"]
        print(f"\nOverall (1-5):")
        print(f"  Fluency:   {avg['fluency']:.2f}")
        print(f"  Coherence: {avg['coherence']:.2f}")
        print(f"  Accuracy:  {avg['accuracy']:.2f}")
        print(f"  Relevance: {avg['relevance']:.2f}")
        print(f"  Overall:   {avg['overall']:.2f}")

---
# 📊 Compare All Results

In [ ]:
import json
import os
from glob import glob

# Find all scoring files
score_files = glob(os.path.join(OUTPUT_DIR, "*_scored_*.json"))

if score_files:
    print(f"{'='*80}")
    print("COMPARISON OF ALL RESULTS")
    print(f"{'='*80}")
    print(f"\n{'Thoth Model':<12} | {'Scorer':<12} | {'Fluency':>8} | {'Coherence':>9} | {'Accuracy':>8} | {'Overall':>8}")
    print("-" * 80)
    
    for f in sorted(score_files):
        with open(f, "r") as file:
            data = json.load(file)
        
        if "overall_averages" in data.get("summary", {}):
            avg = data["summary"]["overall_averages"]
            # Extract names from filename
            basename = os.path.basename(f).replace(".json", "")
            parts = basename.split("_scored_")
            thoth = parts[0]
            scorer = parts[1] if len(parts) > 1 else "unknown"
            
            print(f"{thoth:<12} | {scorer:<12} | {avg['fluency']:>8.2f} | {avg['coherence']:>9.2f} | {avg['accuracy']:>8.2f} | {avg['overall']:>8.2f}")
else:
    print(f"No scoring results found in {OUTPUT_DIR}")

## 📥 Download Results

In [ ]:
from google.colab import files

output_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')]
print(f"Available files:")
for f in output_files:
    print(f"  - {f}")

for f in output_files:
    files.download(os.path.join(OUTPUT_DIR, f))